# 04 — Adaptive Integration + Evaluation

Runs B1 / B2 / B3 / Adaptive on the held-out test split and writes `artifacts/results.csv`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if not (ROOT / 'src').exists():
    ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))
print('ROOT =', ROOT)

In [ ]:
# Use extractive generator for fast offline eval; set --generator hf on Colab GPU for LLM answers
!python "{ROOT}/scripts/evaluate.py" --split test --generator extractive --out "{ROOT}/artifacts/results.csv"

In [ ]:
df = pd.read_csv(ROOT / 'artifacts' / 'results.csv')
summary = df.groupby('system').agg(
    contains=('contains', 'mean'),
    token_f1=('token_f1', 'mean'),
    temporal_acc=('temporal_acc', 'mean'),
    retrieval_rate=('did_retrieve', 'mean'),
    latency_s=('latency_s', 'mean'),
    precision_at_k=('precision_at_k', 'mean'),
).reset_index()
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, col, title in zip(
    axes,
    ['contains', 'temporal_acc', 'retrieval_rate'],
    ['Answer contains@gold', 'Temporal accuracy', 'Retrieval rate'],
):
    axes_df = summary.set_index('system')[col]
    axes_df.plot(kind='bar', ax=ax, color='#2a6f97')
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
fig_path = ROOT / 'artifacts' / 'results_charts.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print('Saved', fig_path)
plt.show()

In [ ]:
# Error analysis slice: current/historical where Adaptive beats / loses to Standard RAG
pivot = df.pivot_table(index='query_id', columns='system', values='temporal_acc')
if set(['adaptive', 'B2_standard_rag']).issubset(pivot.columns):
    pivot['delta'] = pivot['adaptive'] - pivot['B2_standard_rag']
    print('Mean temporal delta Adaptive - B2:', pivot['delta'].mean())
    print(pivot.sort_values('delta', ascending=False).head(10))